# Feature Engineering – NBA Player Longevity Prediction

### Rationale & Feature Engineering Pipeline Choices:
- **Target Definition & Exploration**: The target variable `target_5yrs` is isolated immediately to eliminate data leakage. A distribution countplot is rendered to assess class imbalance.
- **Noise Removal**: `Unnamed: 0` and `name` are removed because they contain high-cardinality unique keys or textual descriptors that act as non-predictive statistical noise.
- **Multicollinearity Control**: Features with an absolute Pearson correlation coefficient $|r| \geq 0.90$ are dropped after visualizing the feature space using a heatmap matrix. This handles structural redundancy (e.g., dropping `fgm` and `fga` because they directly dictate `pts`).
- **Composite Metric Extraction**: Four domain-specific features are engineered (`pts_per_min`, `efficiency_rating`, `ast_tov_ratio`, and `true_shooting_approx`) to evaluate volume, scoring efficiency, and playmaker reliability normalized by runtime minutes.
- **Imputation & Transformation**: Infinite floats resulting from division steps are recast to `NaN` values and imputed using column-wise median values to resist outlier pull.
- **Feature Scaling**: An explicit `StandardScaler` transformation is executed across all feature dimensions to ensure distance-based predictors treat variances uniformly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df_raw = pd.read_csv('0f484464-3cc8-4bfc-968b-b1a9fc4d4b1d.csv')

# Isolate target variable
y = df_raw['target_5yrs'].copy()

print(f"Raw dataset dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw.head()

In [ ]:
# 1. Target Class Balance Visualization
plt.figure(figsize=(6, 4))
sns.countplot(x='target_5yrs', data=df_raw, palette='viridis')
plt.title('Class Balance Evaluation: target_5yrs Distribution')
plt.xlabel('Career Duration >= 5 Years (0 = No, 1 = Yes)')
plt.ylabel('Player Frequency Count')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# Display class summary statistics
counts = df_raw['target_5yrs'].value_counts()
percentages = df_raw['target_5yrs'].value_counts(normalize=True) * 100
for val in [0, 1]:
    print(f"Class {val}: Count = {counts[val]}, Proportion = {percentages[val]:.2f}%")

In [ ]:
# 2. Noise Removal & Initial Column Dropping
DROP_COLS = ['Unnamed: 0', 'name', 'target_5yrs']
df_features = df_raw.drop(columns=DROP_COLS, errors='ignore').copy()
print("Remaining feature columns after removing structural noise keys:")
print(df_features.columns.tolist())

In [ ]:
# 3. Multicollinearity Assessment via Correlation Matrix Heatmap
corr_matrix = df_features.corr().abs()

plt.figure(figsize=(14, 11))
sns.heatmap(df_features.corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Pearson Correlation Coefficient Matrix Heatmap (Raw Features)')
plt.show()

# Remove highly redundant columns with cross-correlations >= 0.90
DROP_REDUNDANT = ['fgm', 'fga', 'ftm', '3pa', 'reb']
df_features.drop(columns=DROP_REDUNDANT, errors='ignore', inplace=True)
print(f"Features remaining following multicollinearity cleanup (Dropped {DROP_REDUNDANT}):")
print(df_features.columns.tolist())

In [ ]:
# 4. Feature Extraction: Creating Composite Performance Metrics
# Points Per Minute
df_features['pts_per_min'] = df_features['pts'] / (df_features['min'] + 1e-9)

# Efficiency Rating (Box Score Metric Normalized per Minute)
df_features['efficiency_rating'] = (df_features['pts'] + df_features['oreb'] + df_features['dreb'] + 
                                     df_features['ast'] + df_features['stl'] + df_features['blk'] - 
                                     df_features['tov']) / (df_features['min'] + 1e-9)

# Assist-to-Turnover Ratio
df_features['ast_tov_ratio'] = df_features['ast'] / (df_features['tov'] + 0.001)

# True Shooting Percentage Approximation
fg_pct = df_features['fg'] / 100
fga_est = np.where(fg_pct > 0, (df_features['pts'] - df_features['3p_made'] * 0.5) / (fg_pct + 1e-9), 0)
fga_est = np.clip(fga_est, 0.5, None)
df_features['true_shooting_approx'] = (df_features['pts'] / (2 * (fga_est + 0.44 * df_features['fta'] + 1e-9))).clip(0, 1)

print("Engineered composite metrics generated successfully. Current feature count:", df_features.shape[1])

In [ ]:
# 5. Handling Missing Values & Edge Case Remapping
df_features.replace([np.inf, -np.inf], np.nan, inplace=True)

# Quantify and execute median imputation for robustness to outliers
null_columns = df_features.columns[df_features.isnull().any()].tolist()
if null_columns:
    print(f"Missing observations found in: {null_columns}. Standardizing using median values.")
    df_features[null_columns] = df_features[null_columns].fillna(df_features[null_columns].median())
else:
    print("No missing value records identified.")

In [ ]:
# 6. Feature Transformation: Standard Feature Scaling
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_features)

# Rebuild DataFrame mapping scaled array back to feature names
df_scaled = pd.DataFrame(scaled_data, columns=df_features.columns)
print("Feature scaling successfully completed.")
print("Verifying distribution means (~0) and standard deviations (~1):")
display(df_scaled.describe().round(4).loc[['mean', 'std']])

In [ ]:
# 7. Format Final ML-Ready Output
df_final = df_scaled.copy()
df_final['target_5yrs'] = y.values

print("Final preprocessed feature set metadata verification:")
df_final.info()
df_final.head()